# 第 6 章 漸近理論: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/econometrics-programming/06-asymptotics.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
seed = 0                         # 乱数の種
sample_sizes = [30, 100, 250, 500]  # 標本の大きさ（比べる）
repetitions = 1000               # 標本を取り直す回数
skewed = True                    # True: 誤差が歪んだ分布（歪度 2）
test_n = 50                      # 検定の標本の大きさ

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

## 標本が大きいと、推定値は真の値（0.5）に集まる

In [ ]:
import numpy as np

from lib.chapter06.asymptotics import (
    consistency_experiment,
    jarque_bera_rejection_rate,
)

estimates = consistency_experiment(
    sample_sizes, repetitions, np.random.default_rng(seed)
)
for size, values in estimates.items():
    print(f"n={size}: 平均 {values.mean():.3f}、標準偏差 {values.std(ddof=1):.3f}")

rate = jarque_bera_rejection_rate(
    test_n, 500, skewed, np.random.default_rng(seed)
)
print(f"Jarque-Bera 検定の棄却率（n={test_n}、歪んだ誤差 {skewed}） {rate:.3f}")

## 図: 標本の大きさごとの推定値の分布

In [ ]:
from lib.chapter06.plot import plot_estimate_distributions

plot_estimate_distributions(estimates, 0.5)

## 試してみよう

- 標本の大きさ 30・100・250・500 で、推定値の標準偏差は 0.197・0.101・0.062・0.045
  と縮み、どれも平均は 0.5 のまわりです（一致性）
- `skewed` を `False` にすると、Jarque-Bera 検定の棄却率は 0.928 から 0.034
  に下がります（正規分布の誤差を、ほとんど棄却しません）
- `test_n` を 500 にすると、歪んだ誤差の棄却率は 1.000 です。標本が大きいほど、
  正規でないことを、見抜けます